In [16]:
import pandas as pd

check_runs_commits = pd.read_json('../datasets/raw/eligible_commits_check_runs.jsonl', lines=True).copy()
check_runs_commits = check_runs_commits.rename(columns={'pr': 'pr_url', 'commit': 'sha'})


In [17]:
check_runs_commits.head()

,sha,pr_url,pr_type,check_runs
0,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,"{'total_count': 44, 'check_runs': [{'id': 4577..."
1,4cbfa276acb89dec117d822b956909e0d54dc51d,https://github.com/umputun/spot/pull/258,ai,"{'total_count': 4, 'check_runs': [{'id': 37903..."
2,5943104753be2820df8bac1fe123644bd3f461e6,https://github.com/nrwl/nx/pull/31380,ai,"{'total_count': 0, 'check_runs': []}"
3,7916be281fa7a9f8e0dfe31e75f8759143881a25,https://github.com/nrwl/nx/pull/31380,ai,"{'total_count': 9, 'check_runs': [{'id': 43216..."
4,0a8bf204f723f81c8df376993827373e19d74e3d,https://github.com/JessicaTegner/pypandoc/pull...,ai,"{'total_count': 1, 'check_runs': [{'id': 44104..."


# Normalize data

In [18]:
normalized_runs = pd.json_normalize(check_runs_commits['check_runs'])
# normalized_runs = normalized_runs.rename(columns=lambda x: f'check_run_{x}')
runs_concat  = pd.concat([check_runs_commits.drop(columns=['check_runs']), normalized_runs], axis=1)
runs_concat.head()

,sha,pr_url,pr_type,total_count,check_runs
0,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,"[{'id': 45774797087, 'name': 'validate-labeled..."
1,4cbfa276acb89dec117d822b956909e0d54dc51d,https://github.com/umputun/spot/pull/258,ai,4,"[{'id': 37903052988, 'name': 'Cloudflare Pages..."
2,5943104753be2820df8bac1fe123644bd3f461e6,https://github.com/nrwl/nx/pull/31380,ai,0,[]
3,7916be281fa7a9f8e0dfe31e75f8759143881a25,https://github.com/nrwl/nx/pull/31380,ai,9,"[{'id': 43216060170, 'name': 'claude', 'node_i..."
4,0a8bf204f723f81c8df376993827373e19d74e3d,https://github.com/JessicaTegner/pypandoc/pull...,ai,1,"[{'id': 44104878698, 'name': 'CodeQL', 'node_i..."


In [19]:
runs_exploded = runs_concat.explode('check_runs').reset_index(drop=True)
runs_exploded['check_runs'] = runs_exploded['check_runs'].apply(
    lambda runs: runs if isinstance(runs, dict) and len(runs) > 0 else {}
)
runs_normalized = pd.json_normalize(runs_exploded['check_runs']).reset_index(drop=True)
runs_normalized = pd.concat([runs_exploded.drop(columns=['check_runs']), runs_normalized], axis=1)
runs_normalized

,sha,pr_url,pr_type,total_count,id,name,node_id,head_sha,external_id,url,...,app.permissions.organization_codespaces_settings,app.permissions.organization_custom_org_roles,app.permissions.organization_events,app.permissions.organization_plan,app.permissions.organization_self_hosted_runners,app.permissions.organization_user_blocking,app.permissions.plan,app.permissions.repository_advisories,app.permissions.team_discussions,app.permissions.organization_secrets
0,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,validate-labeled,CR_kwDOCB5Jx88AAAAKqGP5Hw,9325e444ec074e4c1b636b52c6492217eec7bf23,10642fc2-a53b-5d56-afa0-d1cc32f301dd,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,cancel,CR_kwDOCB5Jx88AAAAKqGP49A,9325e444ec074e4c1b636b52c6492217eec7bf23,7e3678c1-0b48-55cf-8a12-27d31d34a936,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,post-merge,CR_kwDOCB5Jx88AAAAKqGP43w,9325e444ec074e4c1b636b52c6492217eec7bf23,ccb6f45c-73f0-5db4-8257-af0ce25901b5,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577288e+10,test2,CR_kwDOCB5Jx88AAAAKqEapZA,9325e444ec074e4c1b636b52c6492217eec7bf23,e4ac2eee-2c25-5f9b-9684-898d94ebf9de,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577288e+10,test1,CR_kwDOCB5Jx88AAAAKqEapTg,9325e444ec074e4c1b636b52c6492217eec7bf23,c2f0057b-9fc5-51df-b28e-215076d2315f,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137815,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Mintlify Deployment,CR_kwDONq3Itc8AAAAI7PAy9g,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137816,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAytQ,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137817,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAwpg,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137818,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAtyA,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
runs_normalized

,sha,pr_url,pr_type,total_count,id,name,node_id,head_sha,external_id,url,...,app.permissions.organization_codespaces_settings,app.permissions.organization_custom_org_roles,app.permissions.organization_events,app.permissions.organization_plan,app.permissions.organization_self_hosted_runners,app.permissions.organization_user_blocking,app.permissions.plan,app.permissions.repository_advisories,app.permissions.team_discussions,app.permissions.organization_secrets
0,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,validate-labeled,CR_kwDOCB5Jx88AAAAKqGP5Hw,9325e444ec074e4c1b636b52c6492217eec7bf23,10642fc2-a53b-5d56-afa0-d1cc32f301dd,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,cancel,CR_kwDOCB5Jx88AAAAKqGP49A,9325e444ec074e4c1b636b52c6492217eec7bf23,7e3678c1-0b48-55cf-8a12-27d31d34a936,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577480e+10,post-merge,CR_kwDOCB5Jx88AAAAKqGP43w,9325e444ec074e4c1b636b52c6492217eec7bf23,ccb6f45c-73f0-5db4-8257-af0ce25901b5,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577288e+10,test2,CR_kwDOCB5Jx88AAAAKqEapZA,9325e444ec074e4c1b636b52c6492217eec7bf23,e4ac2eee-2c25-5f9b-9684-898d94ebf9de,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,ai,44,4.577288e+10,test1,CR_kwDOCB5Jx88AAAAKqEapTg,9325e444ec074e4c1b636b52c6492217eec7bf23,c2f0057b-9fc5-51df-b28e-215076d2315f,https://api.github.com/repos/mlflow/mlflow/che...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137815,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Mintlify Deployment,CR_kwDONq3Itc8AAAAI7PAy9g,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137816,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAytQ,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137817,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAwpg,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137818,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,https://github.com/trycompai/comp/pull/111,human,13,3.833490e+10,Vercel Preview Comments,CR_kwDONq3Itc8AAAAI7PAtyA,5acbbe47e7c47e6bf888c4b54e733b4f7d0cfdc4,,https://api.github.com/repos/trycompai/comp/ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Per-PR build failure rate = failed valid CI checks/builds / all valid CI checks/builds.

VALID_CONCLUSIONS = {'success', 'failure'}
FAILED_CONCLUSIONS = {'failure'}

pr_keys = ['pr_type', 'pr_url']
commit_keys = pr_keys + ['sha']

# ------------------------------------------------------------------
# 1. Load the complete PR population
# ------------------------------------------------------------------

prs_meta = (
    pd.read_csv(
        '../datasets/processed/rq1/all_eligible_prs.csv',
        usecols=['pr_type', 'pr_url', 'first_commit_sha'],
        dtype={
            'pr_type': 'string',
            'pr_url': 'string',
            'first_commit_sha': 'string',
        },
    )
    .drop_duplicates(subset=['pr_url'])
)

pr_results = prs_meta[pr_keys].copy()

# ------------------------------------------------------------------
# 2. Normalize CI/check-run data
# ------------------------------------------------------------------

failure_metrics = runs_normalized.copy()

failure_metrics['sha'] = failure_metrics['sha'].astype('string')
failure_metrics['pr_url'] = failure_metrics['pr_url'].astype('string')
failure_metrics['pr_type'] = failure_metrics['pr_type'].astype('string')

failure_metrics['status'] = (
    failure_metrics['status']
    .astype('string')
    .str.strip()
    .str.lower()
)

failure_metrics['conclusion'] = (
    failure_metrics['conclusion']
    .astype('string')
    .str.strip()
    .str.lower()
)

# Remove malformed rows without a commit SHA.
failure_metrics = failure_metrics[
    failure_metrics['sha'].notna()
].copy()

# ------------------------------------------------------------------
# 3. Keep only completed, binary CI outcomes
# ------------------------------------------------------------------

valid_ci = failure_metrics[
    failure_metrics['status'].eq('completed')
    & failure_metrics['conclusion'].isin(VALID_CONCLUSIONS)
].copy()

valid_ci['check_failed'] = (
    valid_ci['conclusion'].isin(FAILED_CONCLUSIONS)
)

# Remove duplicated check/job records if the extraction produced any.
valid_ci = valid_ci.drop_duplicates(
    subset=commit_keys + ['id']
)

# Aggregate completed success/failure CI checks (builds) directly to PR level.
pr_build_summary = (
    valid_ci
    .groupby(pr_keys, dropna=False, as_index=False)
    .agg(
        total_ci_builds=('id', 'nunique'),
        failed_ci_builds=('check_failed', 'sum'),
    )
)
pr_build_summary['build_failure_rate'] = (
    pr_build_summary['failed_ci_builds']
    / pr_build_summary['total_ci_builds']
)

# ------------------------------------------------------------------
# 4. Extract the workflow-run ID
# ------------------------------------------------------------------

# Prefer an explicit run_id field when available.
if 'run_id' in valid_ci.columns:
    valid_ci['workflow_run_id'] = (
        valid_ci['run_id'].astype('string')
    )
else:
    valid_ci['workflow_run_id'] = pd.Series(
        pd.NA,
        index=valid_ci.index,
        dtype='string',
    )

# Extract from URLs such as:
# https://github.com/owner/repo/actions/runs/123456/jobs/...
url_run_id = (
    valid_ci['html_url']
    .astype('string')
    .str.extract(r'/actions/runs/(\d+)', expand=False)
    .astype('string')
)

missing_run_id = (
    valid_ci['workflow_run_id'].isna()
    | valid_ci['workflow_run_id'].eq('')
)

valid_ci.loc[missing_run_id, 'workflow_run_id'] = (
    url_run_id.loc[missing_run_id]
)

# Do not fall back to the individual check/job ID.
# Rows without a workflow_run_id can still be used for PR-level
# build failure, but not for workflow-run-level calculations.

# ------------------------------------------------------------------
# 5. Aggregate valid results to commit level
# ------------------------------------------------------------------

commit_level = (
    valid_ci
    .groupby(commit_keys, dropna=False, as_index=False)
    .agg(
        commit_failed=('check_failed', 'any'),
        valid_checks=('id', 'nunique'),
    )
)

commit_level['commit_has_ci'] = True

# ------------------------------------------------------------------
# 6. Aggregate commits to PR level
# ------------------------------------------------------------------

pr_commit_summary = (
    commit_level
    .groupby(pr_keys, dropna=False, as_index=False)
    .agg(
        commits_with_ci=('sha', 'nunique'),
        failed_commits=('commit_failed', 'sum'),
        pr_build_failed=('commit_failed', 'any'),
    )
)

pr_commit_summary['commit_failure_rate'] = (
    pr_commit_summary['failed_commits']
    / pr_commit_summary['commits_with_ci']
)

# Start with all eligible PRs so PRs without CI are retained.
pr_results = pr_results.merge(
    pr_commit_summary,
    on=pr_keys,
    how='left',
    validate='one_to_one',
)

pr_results = pr_results.merge(
    pr_build_summary,
    on=pr_keys,
    how='left',
    validate='one_to_one',
)

for column in [
    'commits_with_ci', 'failed_commits',
    'total_ci_builds', 'failed_ci_builds',
]:
    pr_results[column] = (
        pr_results[column]
        .fillna(0)
        .astype(int)
    )

pr_results['pr_has_ci'] = (
    pr_results['commits_with_ci'] > 0
)

pr_results['pr_build_failed'] = (
    pr_results['pr_build_failed'].astype('boolean')
)

pr_results.loc[
    ~pr_results['pr_has_ci'],
    'pr_build_failed',
] = pd.NA

pr_results.loc[
    ~pr_results['pr_has_ci'],
    'commit_failure_rate',
] = pd.NA

pr_results.loc[
    pr_results['total_ci_builds'].eq(0),
    'build_failure_rate',
] = pd.NA

# ------------------------------------------------------------------
# 7. Aggregate jobs/checks to workflow-run level
# ------------------------------------------------------------------

workflow_level = (
    valid_ci[
        valid_ci['workflow_run_id'].notna()
        & valid_ci['workflow_run_id'].ne('')
    ]
    .groupby(
        commit_keys + ['workflow_run_id'],
        dropna=False,
        as_index=False,
    )
    .agg(
        workflow_failed=('check_failed', 'any'),
        jobs_in_workflow=('id', 'nunique'),
    )
)

workflow_summary = (
    workflow_level
    .groupby(pr_keys, dropna=False, as_index=False)
    .agg(
        total_workflow_runs=('workflow_run_id', 'nunique'),
        failed_workflow_runs=('workflow_failed', 'sum'),
        total_jobs=('jobs_in_workflow', 'sum'),
    )
)

pr_results = pr_results.merge(
    workflow_summary,
    on=pr_keys,
    how='left',
    validate='one_to_one',
)

for column in [
    'total_workflow_runs',
    'failed_workflow_runs',
    'total_jobs',
]:
    pr_results[column] = (
        pr_results[column]
        .fillna(0)
        .astype(int)
    )

pr_results['workflow_failure_rate'] = (
    pr_results['failed_workflow_runs']
    / pr_results['total_workflow_runs']
)

pr_results.loc[
    pr_results['total_workflow_runs'].eq(0),
    'workflow_failure_rate',
] = pd.NA

# ------------------------------------------------------------------
# 8. Determine initial-commit failure
# ------------------------------------------------------------------

initial_commit_results = (
    prs_meta.rename(
        columns={'first_commit_sha': 'sha'}
    )
    .merge(
        commit_level[
            commit_keys
            + ['commit_failed', 'commit_has_ci']
        ],
        on=commit_keys,
        how='left',
        validate='one_to_one',
    )
)

initial_commit_results['initial_commit_failed'] = (
    initial_commit_results['commit_failed']
    .astype('boolean')
)

initial_commit_results['initial_commit_has_ci'] = (
    initial_commit_results['commit_has_ci']
    .fillna(False)
    .astype(bool)
)

initial_commit_results.loc[
    ~initial_commit_results['initial_commit_has_ci'],
    'initial_commit_failed',
] = pd.NA

pr_results = pr_results.merge(
    initial_commit_results[
        pr_keys
        + [
            'initial_commit_has_ci',
            'initial_commit_failed',
        ]
    ],
    on=pr_keys,
    how='left',
    validate='one_to_one',
)

# ------------------------------------------------------------------
# 9. Save PR-level results
# ------------------------------------------------------------------

pr_results = (
    pr_results
    .sort_values(pr_keys)
    .reset_index(drop=True)
)

pr_results.to_csv(
    '../datasets/processed/eligible_pr_failure_rates.csv',
    index=False,
)

In [22]:
pr_results.head(1)

,pr_type,pr_url,commits_with_ci,failed_commits,pr_build_failed,commit_failure_rate,pr_has_ci,total_workflow_runs,failed_workflow_runs,total_jobs,workflow_failure_rate,initial_commit_has_ci,initial_commit_failed
0,ai,https://github.com/3rdIteration/btcrecover/pul...,18,18,True,1.0,True,54,41,624,0.759259,True,True


In [23]:
pr_failure_rates[pr_failure_rates['commits_with_workflows']==0]

,pr_type,pr_url,total_commits,failed_commits,commits_with_workflows,commits_without_workflows,build_failure_rate,total_workflows,failed_workflows,total_jobs,workflow_failure_rate,initial_commit_failed,initial_commit_has_workflows,initial_build_failure_rate
13,ai,https://github.com/AgentOps-AI/agentops/pull/603,16,0,0,16,NaN,0,0,0,NaN,False,False,NaN
14,ai,https://github.com/AgentOps-AI/agentops/pull/605,4,0,0,4,NaN,0,0,0,NaN,False,False,NaN
24,ai,https://github.com/DannyMac180/meta-agent/pull...,1,0,0,1,NaN,0,0,0,NaN,False,False,NaN
25,ai,https://github.com/DannyMac180/meta-agent/pull/29,1,0,0,1,NaN,0,0,0,NaN,False,False,NaN
36,ai,https://github.com/GrimAnticheat/Grim/pull/2138,30,0,0,30,NaN,0,0,0,NaN,NaN,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1841,human,https://github.com/vlm-run/vlmrun-hub/pull/47,2,0,0,2,NaN,0,0,0,NaN,NaN,<NA>,NaN
1842,human,https://github.com/vlm-run/vlmrun-hub/pull/49,1,0,0,1,NaN,0,0,0,NaN,NaN,<NA>,NaN
1843,human,https://github.com/vlm-run/vlmrun-hub/pull/50,1,0,0,1,NaN,0,0,0,NaN,False,False,NaN
1846,human,https://github.com/waldzellai/waldzell-mcp/pul...,1,0,0,1,NaN,0,0,0,NaN,NaN,<NA>,NaN
